# This is a simple example script for 2D x/y stitching using muvis-align and the multiview-stitcher package

In [ ]:
import sys
sys.path.append('..')

from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import vis_utils

from muvis_align.MVSRegistration import MVSRegistration
from muvis_align.image.util import get_sim_physical_size, show_image, extract_sims_from_fused, \
    sims_from_sims_or_msims
from muvis_align.util import print_dict_simple

## Initialise muvis-align, initialise sims, and pre-process

In [ ]:
reg = MVSRegistration(operation='register', input_path='../data/S000/*.zarr', output_path='../../output/', ui='mpl')
reg.init_data()
msims = reg.msims
register_msims, reg_indices, _ = reg.preprocess(msims)
sims = sims_from_sims_or_msims(msims)

for label, sim in zip(reg.file_labels, sims):
    print(label, si_utils.get_origin_from_sim(sim), get_sim_physical_size(sim))

## Initialise registration parameters

In [ ]:
register_params = {
	'pairing': 'orthogonal',
	'transform_type': 'rigid',
	'method': 'sift',
	'gaussian_sigma': 2,
	'normalisation': True,
	'max_keypoints': 5000,
	'inlier_threshold_factor': 0.05,
	'max_trials': 1000,
	'ransac_iterations': 3,
	'n_parallel_pairwise_regs': 1,
}

## Perform registration (using multiview-stitcher)

In [ ]:
%matplotlib inline
results = reg.register(register_msims=register_msims, register_indices=reg_indices, params=register_params)

qualities = {key: value.item() for key, value in results['registration_qualities'].items()}
print('quality')
print_dict_simple(qualities)

## Show registration mapping

In [ ]:
mappings = results['mappings']
for key, mapping in mappings.items():
    print(f'{reg.file_labels[key]}:\n', mapping.sel(t=0).data)

## Visualise registered sims

In [ ]:
%matplotlib inline
sims = sims_from_sims_or_msims(msims)
fig, ax = vis_utils.plot_positions(sims, transform_key=reg.reg_transform_key, use_positional_colors=False, view_labels=reg.file_labels)

## Perform fusion (using multiview-stitcher)

In [ ]:
%matplotlib inline
fused_msim, _ = reg.fuse(msims)

fused_msim

## Output fused result

In [ ]:
%matplotlib inline
fused_sim = extract_sims_from_fused(fused_msim)
show_image(fused_sim[0, 0])

In [ ]:
reg.save('stitching2d', fused_sim)